# DocMind: the complete RAG pipeline

This notebook walks through **ingestion → sparse/dense retrieval → RRF fusion → grounded generation**. It uses the repository's `sample.txt`, a tiny deterministic dense backend, and a fake Claude client, so every cell runs without downloading an embedding model or spending API credits. At the end, an optional cell shows how to switch to the real Anthropic API.

In [1]:
import sys
from pathlib import Path
from pprint import pprint

# Make imports work whether Jupyter was launched from the repository root
# or from notebooks/.
project_root = next((candidate for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
                     if (candidate / 'app').is_dir()), None)
if project_root is None:
    raise RuntimeError('Launch Jupyter from inside the DocMind repository')
sys.path.insert(0, str(project_root))
from app.ingestion import ingest_file
from app.indexing import BM25Index
from app.retrieval import HybridRetriever
from app.generation import answer_question, format_context

source = project_root / 'sample.txt'
result = ingest_file(source, chunk_size=420, chunk_overlap=60)
print(result.modality, len(result.chunks), 'chunks')
pprint(result.chunks[0].__dict__)

text 5 chunks
{'chunk_id': 'c:\\Users\\enosim\\DOCMIND\\sample.txt:chunk-0',
 'metadata': {'chunk_index': '0',
              'file_type': 'txt',
              'source': 'c:\\Users\\enosim\\DOCMIND\\sample.txt'},
 'source': 'c:\\Users\\enosim\\DOCMIND\\sample.txt',
 'text': 'Retrieval-Augmented Generation (RAG) is an AI framework for '
         'improving the quality of LLM-generated responses by grounding the '
         'model on external sources of knowledge. Implementing RAG in an '
         'answering system has two main benefits: It ensures that the model '
         'has access to the most current, reliable facts, and that users have '
         "access to the model's sources, ensuring that its claims can be "
         'checked for accuracy.\n'
         '\n'
         'By'}


Each chunk retains its text, stable ID, source path, and metadata. That metadata is what makes citations possible later.

In [2]:
# Build the sparse index. BM25 is good at exact terms such as 'overlap'.
sparse = BM25Index()
sparse.add_chunks(result.chunks)
query = 'Why do RAG systems use overlapping chunks?'
pprint(sparse.search(query, top_k=3))

[{'chunk_id': 'c:\\Users\\enosim\\DOCMIND\\sample.txt:chunk-3',
  'id': 'c:\\Users\\enosim\\DOCMIND\\sample.txt:chunk-3',
  'metadata': {'chunk_index': '3',
               'file_type': 'txt',
               'source': 'c:\\Users\\enosim\\DOCMIND\\sample.txt'},
  'score': 8.342575329800068,
  'source': 'c:\\Users\\enosim\\DOCMIND\\sample.txt',
  'text': 'manageable pieces known as "chunks".\n'
          '\n'
          'Chunking strategies play a vital role in RAG performance. If chunks '
          'are too large, the system risks retrieving irrelevant information, '
          'diluting the context. If chunks are too small, they may lack the '
          'necessary context to be useful. To mitigate boundary issues where a '
          'crucial sentence is split in half, systems typically employ an '
          'overlap strategy. This means that the end of one'},
 {'chunk_id': 'c:\\Users\\enosim\\DOCMIND\\sample.txt:chunk-0',
  'id': 'c:\\Users\\enosim\\DOCMIND\\sample.txt:chunk-0',
  'metada

In [3]:
# A deterministic dense-like backend for a reproducible tutorial.
# In production, replace this with app.indexing.VectorStore (ChromaDB).
from app.indexing.tokenizer import tokenize

class KeywordDenseDemo:
    def __init__(self, chunks):
        self.chunks = list(chunks)
    def search(self, query, top_k=10, metadata_filter=None):
        q = set(tokenize(query))
        scored = []
        for chunk in self.chunks:
            if metadata_filter and any(chunk.metadata.get(k) != v for k, v in metadata_filter.items()):
                continue
            words = set(tokenize(chunk.text))
            score = len(q & words) / max(len(q), 1)
            if score:
                scored.append((score, chunk))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [{'id': c.chunk_id, 'chunk_id': c.chunk_id, 'text': c.text,
                 'source': c.source, 'metadata': dict(c.metadata), 'score': s}
                for s, c in scored[:top_k]]

dense = KeywordDenseDemo(result.chunks)
hybrid = HybridRetriever(sparse, dense, candidate_k=5)
retrieved = hybrid.retrieve(query, top_k=3)
for item in retrieved:
    print(item['rrf_score'], item['retrieval_sources'], item['chunk_id'])

0.03278688524590164 ['bm25', 'vector'] c:\Users\enosim\DOCMIND\sample.txt:chunk-3
0.03225806451612903 ['bm25', 'vector'] c:\Users\enosim\DOCMIND\sample.txt:chunk-0
0.031746031746031744 ['bm25', 'vector'] c:\Users\enosim\DOCMIND\sample.txt:chunk-2


RRF combines rank positions, not incompatible BM25 scores and vector distances. Chunks found by both backends receive contributions from both lists.

In [4]:
print(format_context(retrieved))

class FakeBlock:
    def __init__(self, text): self.text = text
class FakeResponse:
    content = [FakeBlock('Overlapping chunks preserve context when a sentence crosses a chunk boundary [sample.txt, chunk-1].')]
    usage = {'input_tokens': 10, 'output_tokens': 16}
class FakeMessages:
    def create(self, **kwargs):
        print('Prompt sent to model:', kwargs['messages'][0]['content'][:180] + '...')
        return FakeResponse()
class FakeClaude:
    messages = FakeMessages()

answer = answer_question(query, retrieved, client=FakeClaude(), model='demo-model')
pprint(answer)

<chunk id="1" citation="[c:\Users\enosim\DOCMIND\sample.txt, chunk-3]">
manageable pieces known as "chunks".

Chunking strategies play a vital role in RAG performance. If chunks are too large, the system risks retrieving irrelevant information, diluting the context. If chunks are too small, they may lack the necessary context to be useful. To mitigate boundary issues where a crucial sentence is split in half, systems typically employ an overlap strategy. This means that the end of one
</chunk>

<chunk id="2" citation="[c:\Users\enosim\DOCMIND\sample.txt, chunk-0]">
Retrieval-Augmented Generation (RAG) is an AI framework for improving the quality of LLM-generated responses by grounding the model on external sources of knowledge. Implementing RAG in an answering system has two main benefits: It ensures that the model has access to the most current, reliable facts, and that users have access to the model's sources, ensuring that its claims can be checked for accuracy.

By
</chunk>

<chunk

## Try the real Claude generator (optional)

Set `ANTHROPIC_API_KEY` in `.env` or your environment, then run this cell. `answer_question` sends only the retrieved chunks, enforces citation instructions, and returns the answer, detected citations, model, and usage.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()
# real_answer = answer_question(query, retrieved)
# pprint(real_answer)

### Production dense retrieval

For semantic matching, use:
```python
from app.indexing import VectorStore
dense = VectorStore(persist_directory='data/chroma')
dense.add_chunks(result.chunks)
hybrid = HybridRetriever(sparse, dense)
```
The first operation downloads/loads `all-MiniLM-L6-v2`; the demo backend above intentionally avoids that cost.